# Iskanje primernega TTS modela za projekt SafeSteps

Cilj tega zvezka je primerjati več TTS rešitev in ugotoviti, katera je najbolj primerna za slovenska opozorila. Pri tem ni najpomembnejše, da je glas popolnoma naraven, ampak da je govor **razumljiv, hiter, zanesljiv in uporaben za kratka varnostna sporočila**.
Govorni izhod mora uporabniku hitro in jasno sporočiti, kaj je sistem zaznal: na primer oviro, smer ovire, nevarno razdaljo ali rezultat zaznave slike.

## Namen testiranja več TTS modelov

V aplikaciji potrebujemo TTS za tri glavne primere:

1. **Hover pomoč v GUI-ju**: ko uporabnik nekaj časa drži miško nad gumbom, aplikacija prebere funkcijo gumba.
2. **Opozorila iz ToF senzorja**: če je ovira preblizu, sistem izgovori opozorilo, npr. *Nevarnost, ovira spredaj*.
3. **Rezultati nevronskih mrež**: aplikacija lahko prebere rezultat zaznave slike ali ToF napovedi.

Zaradi uporabnikov s slabovidnostjo mora biti govor čim bolj jasen. Zaradi kasnejše uporabe na prenosni napravi pa je pomembno tudi, ali model lahko deluje brez interneta.

## Kriteriji primerjave

Modele sem primerjala po naslednjih kriterijih:

| Kriterij | Zakaj je pomemben? |
|---|---|
| Podpora slovenščini | Napačna izgovorjava lahko uporabnika zmede. |
| Razumljivost kratkih opozoril | Opozorila morajo biti jasna že ob prvem poslušanju. |
| Zakasnitev | Pri oviri je pomembno, da sistem ne govori prepozno. |
| Offline delovanje | Sistem je bolj zanesljiv, če ni odvisen od interneta. |
| Cena in licenca | Za šolski projekt je boljše, če rešitev ni odvisna od plačljivih API-jev. |
| Naravnost glasu | Pomembno, vendar manj pomembno kot razumljivost. |

# Pregled kandidatov

| Kandidat | Slovenščina | Online/offline | Prednosti | Slabosti | Smiselnost za SafeSteps |
|---|---:|---:|---|---|---|
| **Microsoft Edge/Azure Neural TTS** (`sl-SI-PetraNeural`, `sl-SI-RokNeural`) | dobra | online | naravna slovenska glasova, hiter test z `edge-tts` | odvisnost od interneta/storitve | zelo dober kandidat za demo |
| **Google Cloud TTS** (`sl-SI`) | dobra | online | uradna podpora za slovenščino, kakovostni glasovi | API ključ, cena, internet | kakovosten cloud kandidat |
| **Piper TTS - slovenski glas Artur** | dobra | offline | lokalni neural TTS, primeren za edge naprave | treba preveriti kakovost in namestitev | najboljši offline kandidat |
| **Windows SAPI / pyttsx3** | odvisno od nameščenega glasu | offline | enostaven Python API | Windows pogosto nima slovenskega glasu | uporaben fallback, če je slovenski glas nameščen |
| **eSpeak NG** (`sl`) | osnovna | offline | lahek, odprtokoden, zanesljiv | robotski glas | dober varnostni fallback |
| **gTTS** | tehnično možna, kakovost ni idealna | online | zelo enostaven za uporabo | online, manj primeren za zanesljivo aplikacijo | ni prva izbira |
| **ElevenLabs / Narakeet** | odvisno od glasu | online | zelo naraven govor pri podprtih jezikih | cena, API, možen naglas pri slovenščini | samo če je kakovost dovolj dobra |
| **XTTS-v2 / Coqui** | ne | lokalno | voice cloning in večjezičnost | slovenščina ni podprta | ne |
| **Kokoro TTS** | ne | lokalno | moderni lahki modeli | slovenščina ni podprta | ne |

In [ ]:
from pathlib import Path
import time

AUDIO_DIR = Path("tts_test_outputs")
AUDIO_DIR.mkdir(exist_ok=True)

def output_path(name: str) -> str:
    return str(AUDIO_DIR / name)

## 1. Microsoft Edge TTS

Microsoftovi neuralni glasovi so zelo praktični za testiranje, ker jih lahko uporabimo z neformalnim Python paketom `edge-tts`. Za slovenščino sta zanimiva predvsem:

- `sl-SI-PetraNeural`
- `sl-SI-RokNeural`

Prednost je kakovosten in razumljiv slovenski govor. Slabost je, da rešitev ni lokalna in zato ni idealna za končno napravo, če želimo popolnoma offline delovanje. Za predstavitev projekta pa je to zelo dobra izbira.

In [ ]:
import edge_tts

async def tts_edge(text: str, voice: str = "sl-SI-PetraNeural", filename: str = "edge_test.mp3"):
    start = time.perf_counter()
    communicate = edge_tts.Communicate(text, voice)
    path = output_path(filename)
    await communicate.save(path)
    elapsed_ms = (time.perf_counter() - start) * 1000
    print(f"Shranjeno: {path} ({elapsed_ms:.0f} ms)")
    return path, elapsed_ms

# Primer v Jupyterju:
await tts_edge("Pozor, ovira pred tabo.", voice="sl-SI-PetraNeural", filename="edge_petra.mp3")
await tts_edge("Pozor, ovira pred tabo.", voice="sl-SI-RokNeural", filename="edge_rok.mp3")

In [ ]:
import sounddevice as sd
import soundfile as sf

audio, rate = sf.read("tts_test_outputs/edge_petra.mp3")
sd.play(audio, samplerate=rate)
sd.wait()

In [ ]:
audio, rate = sf.read("tts_test_outputs/edge_rok.mp3")
sd.play(audio, samplerate=rate)
sd.wait()

In [ ]:
# Seznam slovenskih Edge glasov, če želiš preveriti, kateri so trenutno na voljo.

voices = await edge_tts.list_voices()
slovenian_voices = [v for v in voices if v.get("Locale") == "sl-SI"]
[(v["ShortName"], v.get("Gender")) for v in slovenian_voices]

## 2. Piper TTS

Piper je za naš projekt zelo zanimiv, ker je lokalni TTS. To pomeni, da govor lahko deluje brez internetne povezave. Pri sistemu za pomoč uporabniku je to pomembno, ker opozorila ne smejo odpovedati samo zato, ker ni povezave.

In [ ]:
# Prenos slovenskega Piper glasu Artur.

import urllib.request

PIPER_BASE_URL = "https://huggingface.co/rhasspy/piper-voices/resolve/main/sl/sl_SI/artur/medium/"
PIPER_MODEL = Path("sl_SI-artur-medium.onnx")
PIPER_CONFIG = Path("sl_SI-artur-medium.onnx.json")

def download_piper_artur():
    if not PIPER_MODEL.exists():
        urllib.request.urlretrieve(PIPER_BASE_URL + PIPER_MODEL.name, PIPER_MODEL)
    if not PIPER_CONFIG.exists():
        urllib.request.urlretrieve(PIPER_BASE_URL + PIPER_CONFIG.name, PIPER_CONFIG)
    print("Piper Artur model je pripravljen.")

download_piper_artur()

In [ ]:
# Test Piper modela..

def tts_piper_artur(text: str, filename: str = "piper_artur.wav"):
    import numpy as np
    import soundfile as sf
    from piper.voice import PiperVoice

    download_piper_artur()
    start = time.perf_counter()
    voice = PiperVoice.load(str(PIPER_MODEL))

    audio_chunks = []
    for chunk in voice.synthesize(text):
        audio_chunks.append(np.frombuffer(chunk.audio_int16_bytes, dtype=np.int16))

    audio = np.concatenate(audio_chunks)
    path = output_path(filename)
    sf.write(path, audio, 22050)
    elapsed_ms = (time.perf_counter() - start) * 1000
    print(f"Shranjeno: {path} ({elapsed_ms:.0f} ms)")
    return path, elapsed_ms

# Primer:
tts_piper_artur("Nevarnost, ovira levo, razdalja trideset centimetrov.")

In [ ]:
audio, rate = sf.read("tts_test_outputs/piper_artur.wav")
sd.play(audio, samplerate=rate)
sd.wait()

## 3. Google Cloud TTS

Google Cloud TTS ima podporo za `sl-SI` in je kakovosten online kandidat. Prednost je dobra kakovost, slabost pa je odvisnost od API ključa, interneta in morebitnih stroškov.

Za SafeSteps je primeren za primerjavo kakovosti, za končno lokalno napravo pa je manj idealen kot Piper ali drug offline TTS.

In [ ]:
# Potrebno:
# pip install google-cloud-texttospeech
# Nastavljen mora biti GOOGLE_APPLICATION_CREDENTIALS.

from google.cloud import texttospeech

def tts_google_cloud(text: str, filename: str = "google_sl_test.mp3"):
    client = texttospeech.TextToSpeechClient()
    start = time.perf_counter()

    synthesis_input = texttospeech.SynthesisInput(text=text)
    voice = texttospeech.VoiceSelectionParams(language_code="sl-SI")
    audio_config = texttospeech.AudioConfig(audio_encoding=texttospeech.AudioEncoding.MP3)

    response = client.synthesize_speech(
        input=synthesis_input,
        voice=voice,
        audio_config=audio_config,
    )

    path = output_path(filename)
    with open(path, "wb") as f:
        f.write(response.audio_content)

    elapsed_ms = (time.perf_counter() - start) * 1000
    print(f"Shranjeno: {path} ({elapsed_ms:.0f} ms)")
    return path, elapsed_ms

# Primer:
tts_google_cloud("Pozor, ovira pred tabo.")

## 4. pyttsx3 / Windows SAPI

`pyttsx3` sam po sebi ni model, ampak Python knjižnica, ki uporablja sistemske glasove. Slovenščine ne podpira.

Prednost je offline delovanje.

In [ ]:
import pyttsx3

def list_windows_voices():
    engine = pyttsx3.init()
    voices = engine.getProperty("voices")
    for i, voice in enumerate(voices):
        print(i, voice.name, "|", voice.id)

def tts_pyttsx3(text: str, rate: int = 145, voice_index: int | None = None):
    engine = pyttsx3.init()
    engine.setProperty("rate", rate)
    if voice_index is not None:
        voices = engine.getProperty("voices")
        engine.setProperty("voice", voices[voice_index].id)
    engine.say(text)
    engine.runAndWait()

# list_windows_voices()
tts_pyttsx3("Pozor, ovira pred tabo.")

## 5. eSpeak NG kot varnostni fallback

`eSpeak NG` ni zelo naraven, je pa lahek, odprtokoden in deluje lokalno. Pri varnostnem sistemu je lahko uporaben kot zadnja možnost: tudi če boljši TTS ne deluje, lahko sistem še vedno izgovori osnovno opozorilo.

In [ ]:
import subprocess

def tts_espeak(text: str, speed: int = 145):
    subprocess.run([
        "espeak-ng",
        "-v", "sl",
        "-s", str(speed),
        text,
    ], check=False)

# Primer:
tts_espeak("Nevarnost, ovira levo.")

## 6. gTTS

`gTTS` je zelo enostaven za uporabo, vendar za naš projekt ni najboljša izbira. Ne podpira slovenščine, prav tako pa deluje prek spletne storitve, zato ni primeren, če želimo zanesljivo lokalno delovanje. Poleg tega kakovost in odzivni čas nista pod našim nadzorom.

Uporaben je za hitro testiranje, ne pa kot glavna rešitev za SafeSteps.

In [ ]:
from gtts import gTTS

def tts_gtts(text: str, filename: str = "gtts_sl_test.mp3"):
    start = time.perf_counter()
    path = output_path(filename)
    tts = gTTS(text=text, lang="sl")
    tts.save(path)
    elapsed_ms = (time.perf_counter() - start) * 1000
    print(f"Shranjeno: {path} ({elapsed_ms:.0f} ms)")
    return path, elapsed_ms

# Primer:
tts_gtts("Pozor, ovira pred tabo.")

# Sklep

Za SafeSteps je TTS pomemben predvsem kot dostopnostna in varnostna funkcija. Zato izbira modela ni odvisna samo od naravnosti glasu, ampak predvsem od tega, ali uporabnik opozorilo hitro razume.

Trenutno najbolj smiselna rešitev je:

1. **Za offline razvoj:** Piper 

`gTTS`, ElevenLabs in Narakeet so uporabni za primerjavo, vendar niso najboljša glavna rešitev, ker so odvisni od zunanjih storitev, kakovost slovenščine pa ni vedno dovolj predvidljiva.